# 33. Neural Networks: Perceptron and Multilayer Perceptron (MLP)

## Algorithm Category
**Type**: Neural Networks - Feedforward  
**Complexity**: Medium  
**Use Case**: Basic neural network architecture for classification and regression

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the perceptron and its limitations
- Implement a single-layer perceptron from scratch
- Understand multilayer perceptrons (MLPs) and backpropagation
- Build MLPs using scikit-learn and PyTorch
- Visualize decision boundaries and network architecture
- Apply MLPs to classification and regression problems

## Historical Context

Perceptron was developed by Rosenblatt in 1957:
- Rosenblatt, F. (1957): "The Perceptron: A Perceiving and Recognizing Automaton"
- First artificial neuron model
- Foundation for modern neural networks

**Key Papers/References:**
- Rosenblatt, F. (1957). "The Perceptron: A Perceiving and Recognizing Automaton"
- Rumelhart, D.E., et al. (1986). "Learning representations by back-propagating errors"

## When to Use Perceptron and MLP

Perceptron/MLP is appropriate when:
- You have tabular/structured data
- Non-linear relationships in data
- Classification or regression tasks
- Moderate-sized datasets
- Need interpretable neural network
- Good starting point for deep learning

## Theory & Mechanics

### Mathematical Foundation

**Single Perceptron:**
$$y = f(\sum_{i=1}^{n} w_i x_i + b)$$

Where:
- $w_i$: Weights
- $x_i$: Input features
- $b$: Bias
- $f$: Activation function (step, sigmoid, ReLU, etc.)

**Multilayer Perceptron (MLP):**
- Input layer: Receives features
- Hidden layers: Process information
- Output layer: Produces predictions

**Forward Propagation:**
$$h^{(l)} = f(W^{(l)} h^{(l-1)} + b^{(l)})$$

**Backpropagation:**
- Compute gradients using chain rule
- Update weights using gradient descent
- Propagate errors backward through network

### How It Works

1. **Initialize**: Random weights and biases
2. **Forward pass**: Compute activations layer by layer
3. **Compute loss**: Compare predictions with targets
4. **Backward pass**: Calculate gradients
5. **Update weights**: Gradient descent step
6. **Repeat**: Steps 2-5 until convergence

### Key Hyperparameters

- **hidden_layer_sizes**: Number of neurons in each hidden layer
- **activation**: Activation function ('relu', 'tanh', 'logistic')
- **solver**: Optimization algorithm ('adam', 'sgd', 'lbfgs')
- **learning_rate**: Step size for weight updates
- **max_iter**: Maximum iterations
- **alpha**: L2 regularization parameter

### Advantages

- Can learn non-linear patterns
- Universal function approximator (with enough neurons)
- Works with various data types
- Interpretable architecture
- Foundation for deep learning

### Limitations

- Requires careful hyperparameter tuning
- Can overfit easily
- Slow for very large datasets
- May get stuck in local minima
- Requires feature scaling


## Implementation

Let's implement perceptron and MLP.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import pandas as pd  # Pandas: Data manipulation (DataFrames, data analysis)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# Scikit-learn: Machine learning library
from sklearn.datasets import (
    make_classification,  # Generate synthetic classification data
    make_circles,  # Generate circular classification data (non-linear)
    load_iris  # Iris flower classification dataset
)
from sklearn.neural_network import (
    MLPClassifier,  # Multilayer Perceptron for classification
    MLPRegressor  # Multilayer Perceptron for regression
)
from sklearn.model_selection import train_test_split  # Split data into train/test sets
from sklearn.preprocessing import StandardScaler  # Feature scaling
from sklearn.metrics import (
    accuracy_score,  # Calculate accuracy (for classification)
    mean_squared_error  # Calculate MSE (for regression)
)

# ============================================
# IMPORTING OUR HELPER FUNCTIONS
# ============================================

# Our custom utility functions (organized in src/ directory)
from src.models.supervised import (
    split_data,  # Split data into train/test sets
    evaluate_classifier,  # Evaluate classification models
    evaluate_regressor  # Evaluate regression models
)
from src.models.classification import calculate_classification_metrics  # Calculate precision, recall, F1, etc.

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# SIMPLE PERCEPTRON: Implementation from Scratch
# ============================================

# A perceptron is the simplest neural network: a single neuron
# It learns a linear decision boundary by adjusting weights
# This implementation helps understand how neural networks work

class SimplePerceptron:
    """
    Simple Perceptron implementation from scratch.
    
    A perceptron is a single-layer neural network that learns a linear decision boundary.
    It updates weights based on prediction errors.
    """
    
    def __init__(self, learning_rate=0.1, max_iter=1000):
        """
        Initialize perceptron.
        
        Parameters:
        - learning_rate: Step size for weight updates (how much to change weights)
        - max_iter: Maximum number of training iterations
        """
        self.lr = learning_rate  # Learning rate (typically 0.01 to 1.0)
        self.max_iter = max_iter  # Maximum iterations (prevents infinite loops)
    
    def fit(self, X, y):
        """
        Train perceptron on data.
        
        The perceptron learning algorithm:
        1. Initialize weights to zero (or small random values)
        2. For each sample:
           - Make prediction
           - If wrong, update weights: w = w + learning_rate * error * x
        3. Repeat until all samples are correctly classified or max_iter reached
        """
        n_samples, n_features = X.shape  # Get dimensions
        # n_samples: Number of data points
        # n_features: Number of features
        
        # Initialize weights to zero
        # Weights determine how much each feature contributes to the prediction
        self.weights = np.zeros(n_features)  # One weight per feature
        
        # Initialize bias to zero
        # Bias is like an intercept term (allows decision boundary to not pass through origin)
        self.bias = 0
        
        # Convert labels to -1 and 1 (required for perceptron algorithm)
        # Perceptron works with binary labels: -1 (negative class) and +1 (positive class)
        y_binary = np.where(y == 0, -1, 1)
        # np.where(condition, value_if_true, value_if_false)
        # If y == 0, set to -1, else set to 1
        
        # ============================================
        # PERCEPTRON LEARNING ALGORITHM
        # ============================================
        
        # Repeat training for max_iter iterations (or until convergence)
        for _ in range(self.max_iter):
            errors = 0  # Count misclassifications in this iteration
            
            # Process each training sample
            for i in range(n_samples):
                # ============================================
                # FORWARD PASS: Making a Prediction
                # ============================================
                
                # Calculate weighted sum: w1*x1 + w2*x2 + ... + wn*xn + bias
                # np.dot(X[i], self.weights): Dot product (sum of weights × features)
                output = np.dot(X[i], self.weights) + self.bias
                # output: Raw score (can be any number)
                
                # Apply step function (activation)
                # If output >= 0, predict class 1; else predict class -1
                prediction = 1 if output >= 0 else -1
                # This is the decision boundary: output = 0
                
                # ============================================
                # WEIGHT UPDATE: Learning from Mistakes
                # ============================================
                
                # If prediction is wrong, update weights
                if prediction != y_binary[i]:
                    # Perceptron learning rule:
                    # w_new = w_old + learning_rate * true_label * input
                    # This moves the decision boundary toward the correct side
                    self.weights += self.lr * y_binary[i] * X[i]
                    # self.lr: Learning rate (step size)
                    # y_binary[i]: True label (-1 or +1)
                    # X[i]: Input features
                    # If true label is +1 and we predicted -1, weights increase
                    # If true label is -1 and we predicted +1, weights decrease
                    
                    # Update bias similarly
                    self.bias += self.lr * y_binary[i]
                    # Bias moves in the direction of the true label
                    
                    errors += 1  # Count this error
            
            # ============================================
            # CONVERGENCE CHECK: Stop if Perfect
            # ============================================
            
            # If no errors in this iteration, all samples are correctly classified
            if errors == 0:
                break  # Stop training (converged!)
            # Note: Perceptron only converges if data is linearly separable
    
    def predict(self, X):
        """
        Make predictions on new data.
        
        Uses the learned weights and bias to classify samples.
        """
        # Calculate weighted sum for all samples
        output = np.dot(X, self.weights) + self.bias
        # X: Input features (can be single sample or multiple samples)
        # np.dot(): Matrix multiplication (handles both cases)
        
        # Apply step function: output >= 0 → class 1, else → class 0
        return np.where(output >= 0, 1, 0)
        # np.where(condition, value_if_true, value_if_false)
        # Converts back to 0/1 labels (from -1/+1 used during training)

# ============================================
# TESTING THE PERCEPTRON
# ============================================

# Generate simple 2D classification dataset
# make_classification() creates synthetic data with known structure
X_simple, y_simple = make_classification(
    n_samples=100,  # 100 data points
    n_features=2,  # 2 features (for easy visualization)
    n_redundant=0,  # No redundant features
    n_informative=2,  # 2 informative features
    n_clusters_per_class=1,  # One cluster per class (linearly separable)
    random_state=42  # Reproducibility
)

# Scale features (important for perceptron)
X_simple = StandardScaler().fit_transform(X_simple)
# Normalize to mean=0, std=1

# ============================================
# TRAINING THE PERCEPTRON
# ============================================

# Create perceptron
perceptron = SimplePerceptron(learning_rate=0.1, max_iter=1000)
# learning_rate=0.1: Step size for weight updates
# max_iter=1000: Maximum training iterations

# Train the perceptron
perceptron.fit(X_simple, y_simple)  # Learn from data

# ============================================
# EVALUATING THE PERCEPTRON
# ============================================

# Make predictions on training data
y_pred_simple = perceptron.predict(X_simple)  # Class predictions (0 or 1)

# Calculate accuracy
accuracy = accuracy_score(y_simple, y_pred_simple)  # Compare predictions to true labels

print(f"Simple Perceptron Results:")
print(f"  Accuracy: {accuracy:.3f}")  # Display accuracy
print(f"  Weights: {perceptron.weights}")  # Learned weights (one per feature)
print(f"  Bias: {perceptron.bias:.3f}")  # Learned bias

# Interpretation:
# - Weights show how much each feature contributes to the decision
# - Bias shifts the decision boundary
# - If accuracy = 1.0, data is linearly separable (perceptron converged)


In [ ]:
# ============================================
# VISUALIZING PERCEPTRON DECISION BOUNDARY
# ============================================

# Create figure for visualization
plt.figure(figsize=(10, 6))  # Figure size: 10×6 inches

# Scatter plot of data points
# Plot class 0 points
plt.scatter(X_simple[y_simple == 0, 0], X_simple[y_simple == 0, 1], 
           c='red', marker='o', label='Class 0', alpha=0.7)
# X_simple[y_simple == 0, 0]: First feature of class 0 samples
# X_simple[y_simple == 0, 1]: Second feature of class 0 samples
# c='red': Red color
# marker='o': Circle markers
# alpha=0.7: Semi-transparent

# Plot class 1 points
plt.scatter(X_simple[y_simple == 1, 0], X_simple[y_simple == 1, 1], 
           c='blue', marker='s', label='Class 1', alpha=0.7)
# c='blue': Blue color
# marker='s': Square markers

# ============================================
# PLOTTING DECISION BOUNDARY
# ============================================

# Create a mesh (grid) of points covering the entire plot area
# This allows us to visualize the decision boundary
x_min, x_max = X_simple[:, 0].min() - 0.5, X_simple[:, 0].max() + 0.5
y_min, y_max = X_simple[:, 1].min() - 0.5, X_simple[:, 1].max() + 0.5
# Extend range by 0.5 to show boundary clearly

# np.meshgrid() creates a grid of (x, y) coordinates
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                     np.arange(y_min, y_max, 0.1))
# xx and yy are 2D arrays with all (x, y) combinations
# Step size 0.1 creates fine grid

# Predict class for each point in the mesh
# np.c_[xx.ravel(), yy.ravel()] flattens mesh and combines into (x, y) pairs
Z = perceptron.predict(np.c_[xx.ravel(), yy.ravel()])
# Z contains predicted class (0 or 1) for each mesh point

# Reshape predictions back to 2D (same shape as xx and yy)
Z = Z.reshape(xx.shape)

# Fill regions with colors (shows decision boundary)
plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
# contourf() fills areas with different colors
# Different colors show different predicted classes
# alpha=0.3: Semi-transparent (so we can see data points)
# cmap='RdYlBu': Color scheme (Red-Yellow-Blue)

# Label axes
plt.xlabel('Feature 1')  # X-axis: first feature
plt.ylabel('Feature 2')  # Y-axis: second feature
plt.title('Perceptron Decision Boundary')  # Chart title
plt.legend()  # Show legend
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display the plot

# Interpretation:
# - Decision boundary is a straight line (perceptron is linear)
# - Points on one side are classified as class 0, other side as class 1
# - If data is linearly separable, perceptron finds the boundary
# - If data is not linearly separable, perceptron won't converge


## Multilayer Perceptron (MLP)

Let's implement MLP using scikit-learn.


In [ ]:
# ============================================
# LOADING THE DATASET: Iris Classification
# ============================================

# Load Iris dataset (multiclass classification)
iris = load_iris()  # Returns Bunch object
X = iris.data  # Features: flower measurements (150 samples × 4 features)
y = iris.target  # Target: flower species (0, 1, or 2)

print(f"Dataset Shape: {X.shape}")  # Output: (150, 4) - 150 flowers, 4 features
print(f"Classes: {iris.target_names.tolist()}")  # Output: ['setosa', 'versicolor', 'virginica']

# ============================================
# FEATURE SCALING: Critical for Neural Networks
# ============================================

# Neural networks are sensitive to feature scales
# Features on different scales can cause:
# - Slow convergence
# - Gradient problems (some weights update much faster than others)
# - Poor performance
# Scaling ensures all features contribute equally

# StandardScaler normalizes features to mean=0, std=1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# fit_transform() learns scaling from data and applies it

# ============================================
# TRAIN/TEST SPLIT
# ============================================

# Split data into training and test sets
X_train, X_test, y_train, y_test = split_data(X_scaled, y, test_size=0.2, random_state=42)
# 80% for training, 20% for testing

# ============================================
# MODEL CREATION: Multilayer Perceptron (MLP)
# ============================================

# Create MLPClassifier (Multilayer Perceptron for classification)
# MLP has multiple layers of neurons that can learn non-linear patterns

# MLPClassifier parameters:
# hidden_layer_sizes=(10, 5): Architecture of hidden layers
#   - (10, 5) means: 2 hidden layers with 10 and 5 neurons respectively
#   - Input layer: 4 neurons (4 features)
#   - Hidden layer 1: 10 neurons
#   - Hidden layer 2: 5 neurons
#   - Output layer: 3 neurons (3 classes)
#
# activation='relu': Activation function for hidden layers
#   - 'relu': Rectified Linear Unit (f(x) = max(0, x))
#   - Introduces non-linearity (allows learning complex patterns)
#   - Most common for hidden layers
#
# solver='adam': Optimization algorithm
#   - 'adam': Adaptive Moment Estimation (good default)
#   - Other options: 'sgd' (stochastic gradient descent), 'lbfgs'
#
# learning_rate_init=0.001: Initial learning rate
#   - Step size for weight updates
#   - Smaller = slower but more stable
#   - Larger = faster but may overshoot
#
# max_iter=500: Maximum training iterations
#   - Training stops after this many iterations or when converged
#
# random_state=42: Ensures reproducible results
mlp = MLPClassifier(
    hidden_layer_sizes=(10, 5),  # Architecture: 2 hidden layers
    activation='relu',  # Activation function
    solver='adam',  # Optimizer
    learning_rate_init=0.001,  # Learning rate
    max_iter=500,  # Max iterations
    random_state=42  # Reproducibility
)

# ============================================
# MODEL TRAINING: Learning from Data
# ============================================

# .fit() trains the MLP using backpropagation
# The algorithm:
# 1. Forward pass: Compute activations layer by layer
# 2. Calculate loss: Compare predictions to true labels
# 3. Backward pass: Calculate gradients using chain rule
# 4. Update weights: Gradient descent step
# 5. Repeat until convergence or max_iter reached
mlp.fit(X_train, y_train)  # Train the model

print(f"\nMLP Classifier:")
# Architecture: Input → Hidden → Output
print(f"  Architecture: {X.shape[1]} -> {mlp.hidden_layer_sizes} -> {len(iris.target_names)}")
# Example: 4 -> (10, 5) -> 3 (4 inputs, 2 hidden layers, 3 outputs)

print(f"  Activation: {mlp.activation}")  # Activation function used
print(f"  Solver: {mlp.solver}")  # Optimizer used
print(f"  Number of iterations: {mlp.n_iter_}")  # How many iterations it took

# ============================================
# MAKING PREDICTIONS
# ============================================

# .predict() makes predictions using the trained MLP
# Forward pass through all layers to get class predictions
y_pred = mlp.predict(X_test)  # Class predictions (0, 1, or 2)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)  # Compare predictions to true labels
print(f"\nTest Accuracy: {accuracy:.3f}")  # Display accuracy

# ============================================
# EVALUATING MODEL PERFORMANCE
# ============================================

# Evaluate with helper function
results = evaluate_classifier(mlp, X_test, y_test)
# Returns dictionary with accuracy and other metrics

# Calculate detailed classification metrics
metrics = calculate_classification_metrics(y_test.values, y_pred)
# Returns dictionary with: accuracy, precision, recall, F1

print(f"Precision: {metrics['precision']:.3f}, Recall: {metrics['recall']:.3f}, F1: {metrics['f1_score']:.3f}")
# Precision: Of predicted positives, how many were actually positive
# Recall: Of actual positives, how many did we catch
# F1: Harmonic mean of precision and recall (balances both)


## Learning Curve

Let's visualize the loss during training.


In [ ]:
# ============================================
# VISUALIZING TRAINING LOSS: Learning Progress
# ============================================

# Loss curve shows how the model improves during training
# Decreasing loss = model is learning (getting better predictions)

# Check if loss curve is available
# loss_curve_ is only available for certain solvers (like 'adam' or 'sgd')
if hasattr(mlp, 'loss_curve_'):
    # Create figure for loss curve
    plt.figure(figsize=(10, 6))  # Figure size: 10×6 inches
    
    # Plot loss vs iteration
    plt.plot(mlp.loss_curve_)
    # mlp.loss_curve_: Array of loss values (one per iteration)
    # Loss measures prediction error (lower is better)
    
    plt.xlabel('Iteration')  # X-axis: training iteration
    plt.ylabel('Loss')  # Y-axis: loss value
    plt.title('MLP Training Loss Curve')  # Chart title
    plt.grid(True, alpha=0.3)  # Add grid
    
    # Adjust layout
    plt.tight_layout()
    plt.show()  # Display the plot
    
    # Display final loss
    print(f"Final loss: {mlp.loss_curve_[-1]:.4f}")
    # mlp.loss_curve_[-1]: Last element (final loss value)
    
    # Interpretation:
    # - Decreasing curve = model is learning
    # - Flat curve = model converged (not improving)
    # - Increasing curve = something wrong (may need to adjust learning rate)
else:
    print("Loss curve not available (may need to set warm_start=False or use different solver)")
    # Some solvers (like 'lbfgs') don't store loss curve


## Comparing Different Architectures

Let's test different MLP architectures.


In [ ]:
# ============================================
# COMPARING DIFFERENT ARCHITECTURES
# ============================================

# Network architecture (number of layers and neurons) affects performance
# We'll test different architectures to see which works best

# List of architectures to test
architectures = [
    (5,),      # Single hidden layer, 5 neurons (smallest)
    (10,),     # Single hidden layer, 10 neurons
    (10, 5),   # Two hidden layers: 10, then 5 neurons
    (20, 10),  # Two hidden layers: 20, then 10 neurons (largest)
]
# More neurons = more capacity (can learn more complex patterns)
# But also more risk of overfitting

results = []  # Store results for each architecture

# Test each architecture
for arch in architectures:
    # Create MLP with this architecture
    mlp_test = MLPClassifier(
        hidden_layer_sizes=arch,  # Architecture to test
        activation='relu',  # ReLU activation
        solver='adam',  # Adam optimizer
        learning_rate_init=0.001,  # Learning rate
        max_iter=500,  # Max iterations
        random_state=42  # Reproducibility
    )
    
    # Train the model
    mlp_test.fit(X_train, y_train)
    
    # Make predictions
    y_pred_test = mlp_test.predict(X_test)  # Predictions on test set
    
    # Calculate accuracy
    acc = accuracy_score(y_test, y_pred_test)  # Compare predictions to true labels
    
    # Store results
    results.append({
        'architecture': str(arch),  # Architecture as string (for display)
        'accuracy': acc,  # Test accuracy
        'n_layers': len(arch),  # Number of hidden layers
        'total_neurons': sum(arch)  # Total number of neurons
    })
    
    print(f"Architecture {arch}: Accuracy = {acc:.3f}")

# ============================================
# VISUALIZING RESULTS
# ============================================

# Convert results to DataFrame for easier manipulation
results_df = pd.DataFrame(results)

# Create bar plot comparing architectures
plt.figure(figsize=(10, 6))  # Figure size: 10×6 inches

# Bar plot: one bar per architecture
plt.bar(range(len(results_df)), results_df['accuracy'], alpha=0.7)
# range(len(results_df)): X-positions (0, 1, 2, 3)
# results_df['accuracy']: Bar heights (accuracy values)
# alpha=0.7: Semi-transparent bars

plt.xlabel('Architecture')  # X-axis: network architecture
plt.ylabel('Accuracy')  # Y-axis: test accuracy
plt.title('MLP Performance by Architecture')  # Chart title

# Set x-axis labels to architecture names
plt.xticks(range(len(results_df)), results_df['architecture'], rotation=45)
# rotation=45: Rotate labels 45 degrees (easier to read)

plt.grid(True, alpha=0.3, axis='y')  # Add horizontal grid lines

# Adjust layout
plt.tight_layout()
plt.show()  # Display the plot

# Interpretation:
# - Different architectures may perform differently
# - More neurons doesn't always mean better (may overfit)
# - Need to balance capacity with generalization


## Non-Linear Classification

Let's test MLP on non-linearly separable data.


In [ ]:
# ============================================
# NON-LINEAR CLASSIFICATION: Testing MLP's Power
# ============================================

# MLP can learn non-linear decision boundaries (unlike perceptron)
# We'll test it on non-linearly separable data (concentric circles)

# Generate non-linearly separable dataset
# make_circles() creates two concentric circles (one class inside, one outside)
X_circles, y_circles = make_circles(
    n_samples=300,  # 300 data points
    noise=0.1,  # Amount of random noise (0 = perfect circles)
    factor=0.5,  # Distance between inner and outer circle
    random_state=42  # Reproducibility
)
# This data is NOT linearly separable (can't draw a straight line to separate classes)

# Scale features
X_circles_scaled = StandardScaler().fit_transform(X_circles)

# Split data
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_circles_scaled, y_circles, test_size=0.2, random_state=42
)

# ============================================
# TRAINING MLP ON NON-LINEAR DATA
# ============================================

# Create MLP with hidden layers (allows non-linear boundaries)
mlp_circles = MLPClassifier(
    hidden_layer_sizes=(10, 5),  # 2 hidden layers
    activation='relu',  # ReLU activation (non-linear)
    solver='adam',  # Adam optimizer
    learning_rate_init=0.001,  # Learning rate
    max_iter=500,  # Max iterations
    random_state=42  # Reproducibility
)

# Train the model
mlp_circles.fit(X_c_train, y_c_train)  # Learn non-linear pattern

# ============================================
# EVALUATING PERFORMANCE
# ============================================

# Make predictions
y_c_pred = mlp_circles.predict(X_c_test)  # Class predictions (0 or 1)

# Calculate accuracy
acc_circles = accuracy_score(y_c_test, y_c_pred)  # Compare predictions to true labels

print(f"Circles Dataset:")
print(f"  Test Accuracy: {acc_circles:.3f}")  # Display accuracy

# ============================================
# VISUALIZING NON-LINEAR DECISION BOUNDARY
# ============================================

# Create figure for visualization
plt.figure(figsize=(10, 6))  # Figure size: 10×6 inches

# Create mesh for decision boundary visualization
h = 0.02  # Step size for mesh (smaller = finer grid)
x_min, x_max = X_circles[:, 0].min() - 0.5, X_circles[:, 0].max() + 0.5
y_min, y_max = X_circles[:, 1].min() - 0.5, X_circles[:, 1].max() + 0.5
# Extend range by 0.5 to show boundary clearly

# Create grid of points
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Predict class for each point in the mesh
Z = mlp_circles.predict(np.c_[xx.ravel(), yy.ravel()])
# Returns: predicted class (0 or 1) for each mesh point

# Reshape to 2D
Z = Z.reshape(xx.shape)

# Fill regions with colors (shows decision boundary)
plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
# Different colors show different predicted classes
# The boundary between colors is the decision boundary

# Scatter plot of actual data points
plt.scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap='viridis', 
           edgecolors='black', s=50)
# c=y_circles: Color by true class
# edgecolors='black': Black borders for visibility
# s=50: Point size

plt.xlabel('Feature 1')  # X-axis: first feature
plt.ylabel('Feature 2')  # Y-axis: second feature
plt.title('MLP Decision Boundary (Non-Linear)')  # Chart title
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display the plot

# Interpretation:
# - Decision boundary is curved (non-linear) - MLP can learn this!
# - Perceptron would fail on this data (can only draw straight lines)
# - MLP's hidden layers with ReLU activation enable non-linear boundaries
# - This demonstrates the power of multilayer networks


## Validation & Testing

Let's validate the model and compare activation functions.


In [ ]:
# ============================================
# COMPARING ACTIVATION FUNCTIONS
# ============================================

# Activation functions introduce non-linearity into neural networks
# Different activations can affect performance

# List of activation functions to test
activations = ['identity', 'logistic', 'tanh', 'relu']
# - 'identity': Linear (f(x) = x) - no non-linearity
# - 'logistic': Sigmoid (f(x) = 1/(1+e^(-x))) - S-shaped, range (0, 1)
# - 'tanh': Hyperbolic tangent (f(x) = tanh(x)) - S-shaped, range (-1, 1)
# - 'relu': Rectified Linear Unit (f(x) = max(0, x)) - Most common, range [0, ∞)

activation_results = []  # Store results for each activation

# Test each activation function
for act in activations:
    # Create MLP with this activation
    mlp_act = MLPClassifier(
        hidden_layer_sizes=(10, 5),  # Same architecture for fair comparison
        activation=act,  # Activation function to test
        solver='adam',  # Adam optimizer
        learning_rate_init=0.001,  # Learning rate
        max_iter=500,  # Max iterations
        random_state=42  # Reproducibility
    )
    
    # Train the model
    mlp_act.fit(X_train, y_train)
    
    # Make predictions
    y_pred_act = mlp_act.predict(X_test)  # Predictions on test set
    
    # Calculate accuracy
    acc = accuracy_score(y_test, y_pred_act)  # Compare predictions to true labels
    
    # Store results
    activation_results.append({'activation': act, 'accuracy': acc})
    print(f"Activation {act}: Accuracy = {acc:.3f}")

# ============================================
# ASSERTIONS: Automated Validation Checks
# ============================================

# Check 1: MLP should perform better than random guessing
# For 3-class classification, random = 1/3 ≈ 0.333
assert accuracy > 0.5, "MLP should perform better than random"
# If accuracy ≤ 0.5, model is no better than guessing

# Check 2: MLP should handle non-linear data
# Circles dataset is non-linearly separable, but MLP should still learn
assert acc_circles > 0.5, "MLP should handle non-linear data"
# If accuracy ≤ 0.5, model failed to learn the pattern

print("\n✓ Validation checks passed")  # All checks passed!

# Interpretation:
# - ReLU usually works best for hidden layers (most common choice)
# - Identity (linear) usually performs worst (no non-linearity)
# - Logistic and tanh are similar (both S-shaped)
# - Choice of activation can significantly affect performance


## Summary & Key Takeaways

### Key Concepts Learned

1. **Perceptron Basics**
   - Single-layer neural network
   - Linear classifier
   - Can only learn linearly separable patterns
   - Foundation for neural networks

2. **Multilayer Perceptron (MLP)**
   - Multiple layers of neurons
   - Can learn non-linear patterns
   - Universal function approximator
   - Uses backpropagation for training

3. **Key Components**
   - **Weights**: Connections between neurons
   - **Biases**: Offset terms
   - **Activation functions**: Introduce non-linearity (ReLU, sigmoid, tanh)
   - **Loss function**: Measures prediction error
   - **Optimizer**: Updates weights (SGD, Adam, etc.)

4. **Best Practices**
   - Always scale features before training
   - Start with simple architectures
   - Use ReLU activation for hidden layers
   - Monitor loss curve for convergence
   - Use regularization to prevent overfitting

### When to Use Perceptron and MLP

✅ **Good for:**
- Tabular/structured data
- Classification and regression
- Non-linear relationships
- Moderate-sized datasets
- When you need neural network interpretability
- Starting point for deep learning

❌ **Not ideal for:**
- Image data (use CNNs)
- Sequential data (use RNNs)
- Very large datasets (use deep learning)
- When linear models suffice
- Real-time applications (can be slow)

### Next Steps

- Explore **Deep Neural Networks** with more layers
- Try **Convolutional Neural Networks** for images
- Use **Recurrent Neural Networks** for sequences
- Apply **Regularization** techniques (dropout, L2)
- Experiment with **different optimizers** (Adam, RMSprop)
